# Defect Inspection Pipeline: End-to-End Demonstration

This notebook runs the full inspection pipeline on real frames and shows what
each stage contributes. It does not train anything. For the EfficientNet-B4
training run, see `training.ipynb`; the DQN agent is trained by
`scripts/train/train_dqn.py`.

The pipeline has four stages, wired together once in `src/pipeline.py`:

    DQN gate  ->  EfficientNet-B4  ->  LLaMA 3.2 Vision  ->  DefectReport list

1. **DQN gate.** The frame is split into an 8x8 grid of 64 patches. A trained
   Q-network does a greedy rollout over "inspect patch k" and "stop" actions,
   so cheap frames get a few patches and busy frames get more. Without a
   checkpoint the gate falls back to inspecting every patch.
2. **EfficientNet-B4.** Each selected patch is classified as multi-label:
   four independent sigmoid outputs for pothole, longitudinal crack,
   transverse crack, and alligator crack. An all-zero output means "no defect".
3. **LLaMA 3.2 Vision.** For every patch above threshold, the vision model
   writes a three-line report: severity, likely cause, recommended action.
   This runs through Ollama and needs no fine-tuning.
4. **Report assembly.** `build_report` parses the LLaMA text into structured
   fields and bundles everything into a `DefectReport`.

**What you need to run this notebook:**

- `checkpoints/efficientnet_rdd2022.pth` (from `training.ipynb`)
- `checkpoints/dqn.pth` (from `scripts/train/train_dqn.py`), optional
- A running Ollama with `llama3.2-vision` pulled, optional for stage 3
- A few sample frames (RDD2022 test images, or any road photos)

Every stage degrades gracefully if its dependency is missing, so you can run
the notebook with just the EfficientNet checkpoint and still see stages 1, 2,
and 4.

## 0. Imports and setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '..')  # let the notebook find the project root

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from config import (
    set_seeds, SEED,
    GRID_SIZE,
    RDD2022_DIR,
    RDD2022_CHECKPOINT,
    DQN_CHECKPOINT,
)
from src.pipeline import DefectPipeline, InspectionConfig
from src.models.report import CLASS_NAMES, format_summary
from src.models.llama import check_ollama_available
from src.utils.patches import patch_index_to_coords

set_seeds(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'EfficientNet checkpoint present: {RDD2022_CHECKPOINT.exists()}')
print(f'DQN checkpoint present:          {DQN_CHECKPOINT.exists()}')
print(f'Ollama vision model available:   {check_ollama_available()}')

## 1. Load the pipeline

`DefectPipeline` loads every model once in its constructor, then inspects
frames on demand. `model_status()` reports what actually loaded so the rest
of the notebook can adapt.

In [ ]:
use_llama = check_ollama_available()

cfg = InspectionConfig(use_dqn=True, use_llama=use_llama)
pipeline = DefectPipeline(cfg)

print('Model status:')
for stage, status in pipeline.model_status().items():
    print(f'  {stage:13s} {status}')

if pipeline.efficientnet is None:
    raise RuntimeError(
        'EfficientNet-B4 did not load. Run training.ipynb first to '
        'produce checkpoints/efficientnet_rdd2022.pth.'
    )

## 2. Pick sample frames

The notebook looks for frames in a few standard places: the RDD2022 test
split, then a local `sample_frames/` folder you can drop any road photos
into. Point `SAMPLE_DIR` somewhere else if you want.

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}

def find_sample_frames(limit: int = 3) -> list[Path]:
    """Return up to `limit` frame paths from the first source that has any."""
    candidates = [
        RDD2022_DIR / 'Japan' / 'test' / 'images',
        RDD2022_DIR / 'India' / 'test' / 'images',
        Path('..') / 'sample_frames',
        Path('sample_frames'),
    ]
    for folder in candidates:
        if not folder.is_dir():
            continue
        frames = sorted(p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS)
        if frames:
            print(f'Using {len(frames[:limit])} frame(s) from {folder}')
            return frames[:limit]
    raise RuntimeError(
        'No sample frames found. Download RDD2022 (python data/download_rdd2022.py) '
        'or create a sample_frames/ folder with a few road photos.'
    )

frame_paths = find_sample_frames(limit=3)
frame = np.array(Image.open(frame_paths[0]).convert('RGB'))

plt.figure(figsize=(8, 5))
plt.imshow(frame)
plt.title(f'Input frame: {frame_paths[0].name}  ({frame.shape[1]}x{frame.shape[0]})')
plt.axis('off')
plt.show()

## 3. Stage 1: the DQN patch-selection gate

Running the whole pipeline is one call. To see the gate in isolation we call
the pipeline's internal `_select_patches` on the frame, then draw the 8x8
grid with the selected cells filled in. On a frame with few defects a trained
agent stops after a handful of patches; with no checkpoint it selects all 64.

In [ ]:
def show_patch_selection(frame: np.ndarray, selected: list[int], stopped_early: bool) -> None:
    grid = GRID_SIZE
    h, w = frame.shape[:2]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(frame)
    selected_set = set(selected)
    for idx in range(grid * grid):
        x0, y0, x1, y1 = patch_index_to_coords(idx, (h, w), grid)
        picked = idx in selected_set
        ax.add_patch(plt.Rectangle(
            (x0, y0), x1 - x0, y1 - y0,
            fill=picked, alpha=0.30 if picked else 1.0,
            facecolor='tab:orange' if picked else 'none',
            edgecolor='white', linewidth=0.5,
        ))
    order = 'stopped early' if stopped_early else 'full sweep'
    ax.set_title(f'DQN gate selected {len(selected)} / {grid * grid} patches ({order})')
    ax.axis('off')
    plt.show()

pil_frame = Image.fromarray(frame)
selected, stopped_early = pipeline._select_patches(pil_frame)
show_patch_selection(frame, selected, stopped_early)

if pipeline.q_net is None:
    print('Note: no usable DQN checkpoint, so the gate inspected every patch.')
    print('Train one with: python scripts/train/train_dqn.py')

## 4. Stage 2: EfficientNet-B4 patch classification

Now the full pipeline. `inspect()` runs all four stages and returns an
`InspectionResult`. First we look at the classifier's side of it: which
patches fired, for which defect classes, at what confidence.

In [ ]:
result = pipeline.inspect(frame)

print(f'Inspected {result.patches_inspected} / {result.patches_total} patches')
print(f'Defects found: {len(result.reports)}')
print()

if not result.reports:
    print('No patch crossed the defect threshold on this frame. Try another one:')
    print('  frame = np.array(Image.open(frame_paths[1]).convert("RGB"))')
else:
    for r in sorted(result.reports, key=lambda r: -r.confidence):
        print(
            f'  patch ({r.patch_row}, {r.patch_col})  '
            f'{r.class_name:20s}  confidence {r.confidence:.0%}'
        )

In [ ]:
# Show the flagged patch crops with their predicted class.
if result.reports:
    n = len(result.reports)
    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, r in zip(axes, result.reports):
        ax.imshow(r.patch_image)
        ax.set_title(f'{r.class_name}\n{r.confidence:.0%} at ({r.patch_row}, {r.patch_col})', fontsize=9)
        ax.axis('off')
    for ax in axes[n:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 5. Stage 3: LLaMA 3.2 Vision narrative reports

For each flagged patch the vision model gets the crop plus the classifier's
metadata and writes a three-line report. `parse_llama_report` turns that text
into `severity`, `likely_cause`, and `recommended_action` fields on the
`DefectReport`.

If Ollama was not running when the pipeline loaded, `llama_text` holds an
"unavailable" note and the parsed fields stay empty. The classification
result above is unaffected.

In [ ]:
if not result.reports:
    print('No defects on this frame, so no reports to show.')
elif not use_llama:
    print('Ollama vision model not available. Start it with:')
    print('  ollama serve')
    print('  ollama pull llama3.2-vision')
    print('then re-run the notebook from section 1.')
else:
    for i, r in enumerate(result.reports, start=1):
        print(f'--- Report {i}: {r.class_name} at patch ({r.patch_row}, {r.patch_col}) ---')
        print(f'Severity:           {r.severity}')
        print(f'Likely cause:       {r.likely_cause}')
        print(f'Recommended action: {r.recommended_action}')
        print()

## 6. The assembled result

`highlight_patches` draws the boxes, `format_summary` sorts the reports by
severity and prints the block the CLI and dashboard both show. The pipeline
also records per-stage timing.

In [ ]:
fig, (ax_in, ax_out) = plt.subplots(1, 2, figsize=(14, 5))
ax_in.imshow(frame)
ax_in.set_title('Input')
ax_in.axis('off')
ax_out.imshow(result.annotated_image)
ax_out.set_title(f'Annotated: {len(result.reports)} defect(s)')
ax_out.axis('off')
plt.tight_layout()
plt.show()

print(format_summary(result.reports))
print()
print('Per-stage timing:')
for stage, secs in result.per_stage_s.items():
    print(f'  {stage:13s} {secs:.3f} s')
print(f'  {"total":13s} {result.elapsed_s:.3f} s')

## 7. Does the DQN gate actually help?

The gate is only worth its complexity if it finds defects sooner than a naive
sweep. This is the comparison `scripts/eval/evaluate_dqn.py` makes at full
scale (200 episodes); here we run a short version so the notebook stays fast.

For each episode both agents inspect patches on the same image. We track how
many of the image's defects each has found after every k patches, then
average across episodes to get a recall curve. A useful agent reaches high
recall at a lower k than random.

This section needs the DQN checkpoint and the RDD2022 test split. It is
skipped automatically if either is missing.

In [ ]:
import random as _random

def _dqn_vs_random_recall(n_episodes: int = 20):
    from src.models.dqn import QNetwork
    from src.models.dqn_env import PatchInspectionEnv
    from src.datasets.rdd2022 import COUNTRIES

    test_paths: list[Path] = []
    for country in COUNTRIES:
        d = RDD2022_DIR / country / 'test' / 'images'
        if d.exists():
            test_paths.extend(sorted(d.glob('*.jpg')))
    if not test_paths or not DQN_CHECKPOINT.exists():
        return None

    frozen = pipeline.efficientnet
    env = PatchInspectionEnv(
        image_paths=test_paths[:300], model=frozen, device=DEVICE, grid_size=GRID_SIZE,
    )
    q_net = QNetwork(
        state_dim=env.observation_space.shape[0], n_actions=env.action_space.n,
    ).to(DEVICE)
    q_net.load_state_dict(torch.load(DQN_CHECKPOINT, map_location=DEVICE))
    q_net.eval()

    n_patches = env.n_patches

    def recall_curve(cumulative: list[int], total: int) -> list[float]:
        if total == 0:
            return [1.0] * len(cumulative)
        return [c / total for c in cumulative]

    def pad(curve: list[float]) -> list[float]:
        return curve + [curve[-1]] * (n_patches - len(curve))

    dqn_curves, rand_curves = [], []
    while len(dqn_curves) < n_episodes:
        state, _ = env.reset()
        cum, done = [], False
        while not done:
            st = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                action = int(q_net(st).argmax(dim=1).item())
            state, _, term, trunc, info = env.step(action)
            done = term or trunc
            if action != env.stop_action:
                cum.append(info['defects_found'])
        total = info['defects_total']

        _, info_r = env.reset()
        order = list(range(env.n_patches))
        _random.shuffle(order)
        cum_r = []
        for a in order:
            _, _, term, trunc, info_r = env.step(a)
            cum_r.append(info_r['defects_found'])
            if term or trunc:
                break

        if total == 0 or not cum or not cum_r:
            continue
        dqn_curves.append(pad(recall_curve(cum, total)))
        rand_curves.append(pad(recall_curve(cum_r, total)))

    return (
        np.array(dqn_curves).mean(axis=0),
        np.array(rand_curves).mean(axis=0),
        n_patches,
    )

_bench = _dqn_vs_random_recall(n_episodes=20)
if _bench is None:
    print('Skipped: needs checkpoints/dqn.pth and the RDD2022 test split.')
    print('Run the full comparison later with: python scripts/eval/evaluate_dqn.py')
else:
    dqn_mean, rand_mean, n_patches = _bench
    x = range(1, n_patches + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(x, dqn_mean, label='DQN agent', linewidth=2)
    plt.plot(x, rand_mean, label='Random', linewidth=2, linestyle='--')
    plt.axhline(0.8, color='gray', linestyle=':', linewidth=1, label='80% recall')
    plt.xlabel('Patches inspected')
    plt.ylabel('Defect recall')
    plt.title('DQN vs random: defect recall per patches inspected (20 episodes)')
    plt.legend()
    plt.tight_layout()
    plt.show()

    def steps_to(curve, thr=0.8):
        hit = np.where(curve >= thr)[0]
        return int(hit[0]) + 1 if len(hit) else n_patches
    print(f'Patches to 80% recall  DQN: {steps_to(dqn_mean)}   Random: {steps_to(rand_mean)}')

## 8. Out-of-distribution frames

The classifier was fine-tuned on RDD2022 road-damage photos. Autonomous
vehicle camera footage looks different: wider field of view, more sky and
vehicles, different camera height. Running the pipeline on a few such frames
is a qualitative check, not a scored evaluation, since these frames have no
labels.

Drop a few frames into `sample_frames/ood/` (Waymo, nuScenes, dashcam stills)
to use this section. `scripts/waymo_test.py` does the same thing over a whole
directory with summary statistics.

In [ ]:
ood_dir = next((d for d in [Path('..') / 'sample_frames' / 'ood', Path('sample_frames/ood')] if d.is_dir()), None)
ood_frames = []
if ood_dir:
    ood_frames = sorted(p for p in ood_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)[:3]

if not ood_frames:
    print('Skipped: add a few frames to sample_frames/ood/ to run this section.')
else:
    fig, axes = plt.subplots(len(ood_frames), 1, figsize=(9, 5 * len(ood_frames)))
    axes = np.atleast_1d(axes)
    for ax, path in zip(axes, ood_frames):
        img = np.array(Image.open(path).convert('RGB'))
        res = pipeline.inspect(img)
        ax.imshow(res.annotated_image)
        ax.set_title(f'{path.name}: {len(res.reports)} region(s) flagged')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 9. Summary

- One call, `DefectPipeline.inspect(frame)`, runs the gate, the classifier,
  the vision model, and report assembly, and the same object backs both the
  CLI (`scripts/run_inspection.py`) and the Streamlit dashboard.
- The DQN gate trades a fixed 64-patch sweep for an adaptive one; section 7
  shows whether that trade pays off on held-out frames.
- EfficientNet-B4 does multi-label classification per patch, so one patch can
  carry a pothole and a crack at once.
- LLaMA 3.2 Vision turns each detection into an inspector-style report with no
  training, and its absence never blocks the detection path.